# Part 1 — Scope, Ownership, and Outputs

### Objective

Build the local analysis-ready dataset for dynamic AKI prediction. This notebook owns source validation, cohort construction, KDIGO onset timelines, hourly snapshots, multi-horizon labels, leakage-safe features, and patient-level splits.

### Outputs

- Eligible ICU cohort
- Hourly snapshot dataset
- Patient-level split manifest
- Feature dictionary and aggregate quality report

### Rules

Do not train models here, upload MIMIC data, or commit patient-level artifacts.


# Part 2 — Configuration and Study Definitions

### Objective

Load all study choices from configuration before processing data.

### Required settings

- Local MIMIC-III and artifact roots
- Monitoring start and end times
- Training snapshot interval: 1 hour
- Prediction horizons: 12, 24, and 48 hours
- Primary horizon: 48 hours
- Feature lookback windows
- Follow-up and exclusion rules
- Patient split proportions and random seed
- AKI and baseline-creatinine definition versions

### Rule

Do not redefine these choices in later cells without creating a new dataset version.


# Part 3 — Load and Validate MIMIC-III Tables

### Objective

Load only required columns and standardize identifiers, timestamps, units, and missing values.

### Candidate sources

PATIENTS, ADMISSIONS, ICUSTAYS, LABEVENTS, CHARTEVENTS, OUTPUTEVENTS, and approved diagnosis, procedure, medication, and intervention tables.

### Checks

- subject_id, hadm_id, and icustay_id relationships
- Source counts and timestamp ranges
- CareVue and MetaVision item mappings
- Duplicate, invalid, and incompatible-unit measurements

### Output

Validated local source views with explicit schemas.


# Part 4 — Define the Base ICU Cohort

### Objective

Create the eligible population before repeated snapshots are generated.

### Inclusion rules

- Adults under the prespecified MIMIC age rule
- First ICU stay per patient
- Sufficient follow-up for at least one prediction horizon

### Exclusion rules

- ESKD or chronic dialysis under prespecified definitions
- Invalid ICU timing
- Additional exclusions documented before use

### Output

One row per eligible ICU stay.


# Part 5 — Define Baseline Creatinine and Renal History

### Objective

Create a chronological creatinine record and assign the primary baseline used for KDIGO assessment.

### Rules

- Standardize units and preserve measurement provenance.
- Define how pre-ICU, admission, nadir, and missing baselines are handled.
- Do not use future measurements to construct a snapshot's input features.
- Record CKD, prior renal replacement therapy, and exclusion evidence separately.

### Sensitivity plan

Prepare alternative defensible baseline definitions for later robustness analysis rather than silently choosing the most favorable result.


# Part 6 — Generate KDIGO AKI Onset Timelines

### Objective

Identify the earliest AKI onset, stage, and supporting criterion.

### Primary definition

Creatinine increase of at least 0.3 mg/dL within 48 hours or at least 1.5 times the applicable baseline within the prespecified KDIGO period.

### Robustness extension

Add urine-output KDIGO using documented body weight, rolling windows, missing-output handling, and renal replacement therapy rules.

### Checks

Use manually constructed positive, negative, boundary, duplicate-time, missing-baseline, and low-urine-output cases.


# Part 7 — Generate Hourly Training Snapshots

### Objective

Create one candidate prediction cutoff per ICU hour during the configured monitoring period.

### Rules

- feature_time <= snapshot_time
- Stop snapshots at AKI onset, ICU discharge, death, or monitoring end.
- Record whether each horizon has sufficient outcome follow-up.
- Keep actual AKI-relevant EHR event times for later event-driven replay.
- Do not create minute-level duplicate training rows when no relevant data changed.

### Output

One eligible row per icustay_id and snapshot_time.


# Part 8 — Assign 12h, 24h, and 48h AKI Targets

### Objective

For every snapshot, assign horizon-specific eligibility and labels.

### Label rule

For horizon H, the positive window is (snapshot_time, snapshot_time + H]. A patient with AKI at or before snapshot_time is not at risk and must not contribute a later snapshot.

### Required fields

- eligible_12h, aki_within_12h
- eligible_24h, aki_within_24h
- eligible_48h, aki_within_48h
- aki_onset_time when applicable

Overlapping labels across hours and horizons are expected and are not leakage.


# Part 9 — Extract Leakage-Safe Multimodal Features

### Objective

Summarize information available by each snapshot.

### Modalities

- Demographics and comorbidities
- Vital signs and laboratory measurements
- Glasgow Coma Scale
- Urine output
- Medications and interventions when feasible

### Summaries

Use prespecified latest, mean, minimum, maximum, range, slope, variability, count, missingness, and time-since-last-measurement features over clinically meaningful windows.

### Caution

Measurement frequency can encode workflow and illness severity; retain it only when intentional and test its transportability.


# Part 10 — Build the Dynamic Modeling Table

### Objective

Join all snapshot-specific features and targets into a stable schema.

### Required fields

- subject_id, hadm_id, and icustay_id
- snapshot_time and continuous hours_since_icu
- Horizon-specific labels and eligibility flags
- Feature columns and provenance version
- CareVue or MetaVision source indicator for robustness analysis

### Rules

Keep identifiers, outcomes, onset times, and future information out of the model feature list. Preserve 8h, 12h, and 24h as reporting checkpoints only.


# Part 11 — Create Patient-Level Development Splits

### Objective

Assign each subject_id to exactly one reproducible train, validation, or test split.

### Use

- Train: fitting and patient-grouped internal cross-validation
- Validation: model comparison, calibration decisions, monitoring-window selection, horizon selection, threshold selection, and alert-policy selection
- Test: one locked final evaluation

### Temporal generalization note

MIMIC-III calendar years are patient-shifted and must not be treated as true chronology. Use CareVue versus MetaVision as a system-era robustness analysis, and reserve true temporal or external validation for a database with valid provenance.


# Part 12 — Validate and Export Dataset Artifacts

### Objective

Run final integrity checks and write versioned local artifacts.

### Required checks

- Patient disjointness across splits
- No feature event after its snapshot
- No snapshot at or after AKI onset
- Correct horizon boundaries and follow-up eligibility
- Unique snapshot keys
- Outcome prevalence and sample counts by split, hour, horizon, ICU type, and source system
- Reconciliation of known 8h, 12h, and 24h cohort counts

### Outputs

Export the cohort, snapshot dataset, split manifest, feature dictionary, configuration, and aggregate quality report. Never commit patient-level outputs.
